# HTTP latency analysis for ATOM Sumo

This notebook analyzes the currently available HTTP latency measurements for the robot control interface.

Included runs:
- Run 1: before the motor A wiring fix, speed set to `0`
- Run 2: after the motor A wiring fix, speed set to `0`
- Run 3: after the fix, medium motor speed set to `180`

Important limitation:
- this is **control-path latency** measured over HTTP request-response time;
- this is **not yet** a full mechanical motor test;
- the chassis was not assembled during these runs, so no wheel speed, current, or push-force was measured here.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.float_format = lambda x: f'{x:.3f}'

cwd = Path.cwd()
analysis_dir = cwd if (cwd / 'results').exists() else cwd / 'analysis'
results_dir = analysis_dir / 'results'

runs = [
    {
        'label': 'Run 1: before fix, speed=0',
        'file': 'http_latency_2026-04-11_18-28-33.csv',
        'phase': 'before_fix',
        'speed_value': 0,
    },
    {
        'label': 'Run 2: after fix, speed=0',
        'file': 'http_latency_2026-04-11_19-34-31.csv',
        'phase': 'after_fix_idle',
        'speed_value': 0,
    },
    {
        'label': 'Run 3: after fix, speed=180',
        'file': 'http_latency_2026-04-11_19-38-18_medium_speed.csv',
        'phase': 'after_fix_medium_speed',
        'speed_value': 180,
    },
]

frames = []
for run in runs:
    df = pd.read_csv(results_dir / run['file'])
    df['latency_ms'] = df['latency_ms'].astype(float)
    for key, value in run.items():
        if key != 'file':
            df[key] = value
    frames.append(df)

latency = pd.concat(frames, ignore_index=True)
latency.head()


## Method

Each run alternates the following endpoints for `120` total requests:
- `/move?dir=forward`
- `/move?dir=stop`

Latency is measured from request start until the full HTTP response is received.

This notebook focuses on:
- overall latency summary;
- endpoint-level asymmetry between `forward` and `stop`;
- improvement after the motor A wiring fix;
- extra latency introduced when the motors are enabled at medium speed.


In [ ]:
def p95(series):
    return series.quantile(0.95)


overall_summary = (
    latency.groupby(['label', 'phase', 'speed_value'], sort=False)['latency_ms']
    .agg(
        n='count',
        mean_ms='mean',
        median_ms='median',
        p95_ms=p95,
        max_ms='max',
        std_ms='std',
    )
    .reset_index()
)

overall_summary


In [ ]:
endpoint_summary = (
    latency.groupby(['label', 'endpoint'], sort=False)['latency_ms']
    .agg(
        n='count',
        mean_ms='mean',
        median_ms='median',
        p95_ms=p95,
        max_ms='max',
    )
    .reset_index()
)

endpoint_summary


In [ ]:
targets = pd.DataFrame(
    {
        'metric': ['median_ms', 'p95_ms'],
        'target_ms': [100.0, 150.0],
    }
)

compliance = overall_summary[['label', 'median_ms', 'p95_ms']].copy()
compliance['median_target_met'] = compliance['median_ms'] <= 100.0
compliance['p95_target_met'] = compliance['p95_ms'] <= 150.0
compliance


## Plot 1: Overall summary metrics by run

This plot compares the key aggregate metrics for the three available runs.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
metrics = [
    ('median_ms', 'Median latency (ms)'),
    ('p95_ms', '95th percentile latency (ms)'),
    ('max_ms', 'Max latency (ms)'),
]
colors = ['#b91c1c', '#15803d', '#1d4ed8']

for axis, (metric, title) in zip(axes, metrics):
    axis.bar(overall_summary['label'], overall_summary[metric], color=colors)
    axis.set_title(title)
    axis.set_ylabel('ms')
    axis.tick_params(axis='x', rotation=20)
    for idx, value in enumerate(overall_summary[metric]):
        axis.text(idx, value + 2, f'{value:.1f}', ha='center', va='bottom', fontsize=9)

plt.show()


## Plot 2: Distribution of request latencies by run

The boxplot shows how wide or tight the latency distribution is in each run.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = overall_summary['label'].tolist()
series = [latency.loc[latency['label'] == label, 'latency_ms'] for label in labels]

box = ax.boxplot(series, labels=labels, patch_artist=True, showmeans=True)
for patch, color in zip(box['boxes'], ['#fecaca', '#bbf7d0', '#bfdbfe']):
    patch.set_facecolor(color)

ax.set_title('HTTP latency distribution by run')
ax.set_ylabel('Latency (ms)')
ax.tick_params(axis='x', rotation=20)
plt.show()


## Plot 3: Endpoint asymmetry

This comparison is especially important because the earlier failure mode mostly affected forward motion.


In [ ]:
pivot_median = endpoint_summary.pivot(index='label', columns='endpoint', values='median_ms')
pivot_p95 = endpoint_summary.pivot(index='label', columns='endpoint', values='p95_ms')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
pivot_median.plot(kind='bar', ax=axes[0], color=['#2563eb', '#f97316'])
pivot_p95.plot(kind='bar', ax=axes[1], color=['#2563eb', '#f97316'])

axes[0].set_title('Median latency by endpoint')
axes[0].set_ylabel('ms')
axes[0].tick_params(axis='x', rotation=20)

axes[1].set_title('P95 latency by endpoint')
axes[1].set_ylabel('ms')
axes[1].tick_params(axis='x', rotation=20)

for axis in axes:
    axis.legend(title='Endpoint')

plt.show()


## Plot 4: Request-by-request latency trace

This view helps spot spikes and whether they are isolated or persistent.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for label, group in latency.groupby('label', sort=False):
    ax.plot(group['n'], group['latency_ms'], marker='o', markersize=2.5, linewidth=1, alpha=0.75, label=label)

ax.set_title('Request-by-request HTTP latency')
ax.set_xlabel('Request number')
ax.set_ylabel('Latency (ms)')
ax.legend()
plt.show()


## Plot 5: Histograms by run

The histograms show how concentrated or spread the results are for each operating condition.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True, sharey=True)

for axis, label, color in zip(
    axes,
    overall_summary['label'],
    ['#dc2626', '#16a34a', '#2563eb'],
):
    subset = latency.loc[latency['label'] == label, 'latency_ms']
    axis.hist(subset, bins=15, color=color, alpha=0.8, edgecolor='black')
    axis.set_title(label)
    axis.set_xlabel('Latency (ms)')

axes[0].set_ylabel('Request count')
plt.show()


## Derived comparisons

This cell prints the most useful interpretation values for a report or defense.


In [ ]:
run1 = overall_summary.iloc[0]
run2 = overall_summary.iloc[1]
run3 = overall_summary.iloc[2]

median_improvement_after_fix = 100.0 * (run1['median_ms'] - run2['median_ms']) / run1['median_ms']
median_increase_medium_speed = 100.0 * (run3['median_ms'] - run2['median_ms']) / run2['median_ms']

print(f"Median latency improvement after fix: {median_improvement_after_fix:.1f}%")
print(f"Median latency increase from speed=0 to speed=180: {median_increase_medium_speed:.1f}%")
print()
print('Most important observations:')
print('- The wiring fix removed the very large latency asymmetry seen in the first run.')
print('- After the fix at speed=0, forward and stop latencies are closely matched.')
print('- At speed=180, stop remains fast, while forward becomes the dominant source of additional latency.')
print('- The speed=180 run is still usable for manual operation, but it sits close to the stricter V1 p95 target.')


## Final interpretation

Suggested report language based on the current data:

- The initial pre-fix measurement showed a severe control-path asymmetry, especially for `/move?dir=forward`.
- After correcting the motor A channel issue, idle HTTP control latency improved strongly and became both low and stable.
- A realistic medium-speed command (`speed=180`) increases total request-response time mainly on `forward`, while `stop` remains consistently responsive.
- Therefore the control stack is already good enough for V2 prototyping, but full system validation still requires a physical motor test with assembled chassis, current measurement, and real motion/braking data.
